<a href="https://colab.research.google.com/github/Sounak-thegeek/Blog-Website/blob/master/FakeNewsClassifierUsingLSTM_and_Bi_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("saurabhshahane/fake-news-classification")

print("Path to dataset files:", path)

100%|██████████| 92.1M/92.1M [00:03<00:00, 28.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/saurabhshahane/fake-news-classification/versions/77


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(path+"/WELFake_Dataset.csv", on_bad_lines = 'skip', engine = 'python')

In [ ]:
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [ ]:
df.shape

(72154, 4)

In [ ]:
df.isnull().sum()

,0
Unnamed: 0,0
title,565
text,57
label,20


In [ ]:
df = df.dropna()
df['label'] = pd.to_numeric(df['label'], errors='coerce')
df = df.dropna(subset=['label'])

In [ ]:
X = df.drop('label', axis = 1)

In [ ]:
y = df['label']

In [ ]:
y.value_counts()

,count
label,
1.0,36507
0.0,35028


In [ ]:
X.shape

(71535, 3)

In [ ]:
y.shape

(71535,)

In [ ]:
import tensorflow as tf

In [ ]:
tf.__version__

'2.20.0'

In [ ]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [ ]:
voc_size = 5000

In [ ]:
##One Hot Representation

In [ ]:
messages = X.copy()

In [ ]:
messages['title'][0]

'LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO]'

In [ ]:
messages.reset_index(inplace=True)

In [ ]:
messages

,index,Unnamed: 0,title,text
0,0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...
1,2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ..."
2,3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...
3,4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will..."
4,5,5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...
...,...,...,...,...
71530,72149,72129,Russians steal research on Trump in hack of U....,WASHINGTON (Reuters) - Hackers believed to be ...
71531,72150,72130,WATCH: Giuliani Demands That Democrats Apolog...,"You know, because in fantasyland Republicans n..."
71532,72151,72131,Migrants Refuse To Leave Train At Refugee Camp...,Migrants Refuse To Leave Train At Refugee Camp...
71533,72152,72132,Trump tussle gives unpopular Mexican leader mu...,MEXICO CITY (Reuters) - Donald Trump’s combati...


In [ ]:
import nltk
import re
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
  review = re.sub('[^a-zA-Z0-9]', ' ', messages['title'][i])
  review = review.lower()
  review = review.split()

  review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
  review = ' '.join(review)
  corpus.append(review)

In [ ]:
corpus

['law enforc high alert follow threat cop white 9 11bi blacklivesmatt fyf911 terrorist video',
 'unbeliev obama attorney gener say charlott rioter peac protest home state north carolina video',
 'bobbi jindal rais hindu use stori christian convers woo evangel potenti 2016 bid',
 'satan 2 russia unv imag terrifi new supernuk western world take notic',
 'time christian group sue amazon splc design hate group',
 'dr ben carson target ir never audit spoke nation prayer breakfast',
 'hous intel chair trump russia fake stori evid anyth video',
 'sport bar owner ban nfl game show true american sport like speak rural america video',
 'latest pipelin leak underscor danger dakota access pipelin',
 'gop senat smack punchabl alt right nazi internet',
 'may brexit offer would hurt cost eu citizen eu parliament',
 'schumer call trump appoint offici overse puerto rico relief',
 'watch hilari ad call question health age clinton crime famili boss',
 'chang expect espn polit agenda despit huge subscrib 

In [ ]:
onehot_repr=[one_hot(words, voc_size) for words in corpus]
onehot_repr

[[1414,
  4205,
  101,
  333,
  4007,
  3945,
  2195,
  3785,
  4333,
  1208,
  4706,
  1713,
  1277,
  4749],
 [1740,
  3134,
  4739,
  4289,
  2013,
  3233,
  172,
  562,
  2700,
  623,
  3098,
  545,
  1024,
  4749],
 [269, 4446, 2730, 3507, 113, 1217, 1855, 382, 784, 1513, 4701, 955, 600],
 [3966, 2215, 2220, 4349, 4159, 3556, 1072, 1340, 475, 3038, 200, 3913],
 [849, 1855, 4109, 3325, 4070, 1834, 1929, 766, 4109],
 [3235, 3745, 4322, 4126, 2692, 298, 3944, 627, 3892, 911, 4917],
 [4022, 2660, 1984, 1721, 2220, 2517, 1217, 897, 4015, 4749],
 [609,
  1628,
  535,
  3830,
  2859,
  1238,
  4714,
  4817,
  3540,
  609,
  3171,
  586,
  1589,
  3096,
  4749],
 [2969, 2123, 3491, 592, 2969, 4612, 3453, 2123],
 [4328, 4030, 3223, 97, 443, 1691, 4736, 2997],
 [1462, 1492, 1815, 3408, 2139, 1676, 1054, 1099, 1054, 4882],
 [4691, 4672, 1721, 631, 3805, 4673, 4468, 3686, 413],
 [4041, 1301, 942, 4672, 3531, 1474, 888, 4860, 1238, 3577, 2860],
 [1153, 4940, 3559, 4008, 1855, 2202, 4187, 3491,

In [ ]:
corpus[1]

'unbeliev obama attorney gener say charlott rioter peac protest home state north carolina video'

In [ ]:
onehot_repr[1]

[1740,
 3134,
 4739,
 4289,
 2013,
 3233,
 172,
 562,
 2700,
 623,
 3098,
 545,
 1024,
 4749]

In [ ]:
##Embedding Representation

In [ ]:
sent_length = max(len(sen) for sen in corpus)

In [ ]:
sent_length

299

In [ ]:
embedded_docs = pad_sequences(onehot_repr, padding='pre', maxlen = sent_length)

In [ ]:
print(embedded_docs)

[[   0    0    0 ... 1713 1277 4749]
 [   0    0    0 ...  545 1024 4749]
 [   0    0    0 ... 4701  955  600]
 ...
 [   0    0    0 ... 3919 4635 1802]
 [   0    0    0 ... 4459 4503 1547]
 [   0    0    0 ... 2950 4860 1537]]


In [ ]:
embedded_docs[1]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,   

In [ ]:
from tensorflow.keras.layers import Dropout
embedding_vector_features = 40
model = Sequential()
model.add(Embedding(voc_size, embedding_vector_features, input_length = sent_length))
model.add(Dropout(0.3))
model.add(LSTM(100))
model.add(Dropout(0.3))
model.add(Dense(1, activation = 'sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics =['accuracy'])
print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
len(embedded_docs), y.shape

(71535, (71535,))

In [ ]:
import numpy as np
X_final = np.array(embedded_docs)
y_final = np.array(y)

In [ ]:
X_final.shape, y_final.shape

((71535, 299), (71535,))

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

In [ ]:
##Model training

In [ ]:
y_train = y_train.astype(int)
y_test = y_test.astype(int)
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs = 10, batch_size=32)

Epoch 1/10
1498/1498 ━━━━━━━━━━━━━━━━━━━━ 45s 30ms/step - accuracy: 0.9354 - loss: 0.1589 - val_accuracy: 0.8902 - val_loss: 0.2893
Epoch 2/10
1498/1498 ━━━━━━━━━━━━━━━━━━━━ 45s 30ms/step - accuracy: 0.9451 - loss: 0.1349 - val_accuracy: 0.8885 - val_loss: 0.3314
Epoch 3/10
1498/1498 ━━━━━━━━━━━━━━━━━━━━ 83s 31ms/step - accuracy: 0.9511 - loss: 0.1214 - val_accuracy: 0.8910 - val_loss: 0.3328
Epoch 4/10
1498/1498 ━━━━━━━━━━━━━━━━━━━━ 81s 30ms/step - accuracy: 0.9559 - loss: 0.1120 - val_accuracy: 0.8912 - val_loss: 0.3434
Epoch 5/10
1498/1498 ━━━━━━━━━━━━━━━━━━━━ 44s 30ms/step - accuracy: 0.9608 - loss: 0.1006 - val_accuracy: 0.8914 - val_loss: 0.3877
Epoch 6/10
1498/1498 ━━━━━━━━━━━━━━━━━━━━ 45s 30ms/step - accuracy: 0.9620 - loss: 0.0945 - val_accuracy: 0.8905 - val_loss: 0.3617
Epoch 7/10
1498/1498 ━━━━━━━━━━━━━━━━━━━━ 44s 29ms/step - accuracy: 0.9662 - loss: 0.0854 - val_accuracy: 0.8928 - val_loss: 0.4181
Epoch 8/10
1498/1498 ━━━━━━━━━━━━━━━━━━━━ 45s 30ms/step - accuracy: 0.9685 -

In [ ]:
##Performance metrics and accuracy

In [ ]:
y_pred = model.predict(X_test)

738/738 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step


In [ ]:
X_test

array([[   0,    0,    0, ..., 4109, 1316, 1057],
       [   0,    0,    0, ..., 1876, 4629, 3282],
       [   0,    0,    0, ..., 1466, 4903, 1921],
       ...,
       [   0,    0,    0, ..., 3082,   55, 3735],
       [   0,    0,    0, ..., 3485,  955, 1273],
       [   0,    0,    0, ..., 3096, 3460,  866]], dtype=int32)

In [ ]:
X_test.shape

(23607, 299)

In [ ]:
type(y_pred)

numpy.ndarray

In [ ]:
y_pred.shape

(23607, 1)

In [ ]:
y_pred = np.where(y_pred > 0.5, 1, 0)

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
confusion_matrix(y_test, y_pred)

array([[10417,  1218],
       [ 1400, 10572]])

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.8891006904731648

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.90      0.89     11635
           1       0.90      0.88      0.89     11972

    accuracy                           0.89     23607
   macro avg       0.89      0.89      0.89     23607
weighted avg       0.89      0.89      0.89     23607



In [ ]:
##Bi-directional LSTM

In [ ]:
from tensorflow.keras.layers import Dropout, Bidirectional
embedding_vector_features = 40
model = Sequential()
model.add(Embedding(voc_size, embedding_vector_features, input_length = sent_length))
model.add(Dropout(0.3))
model.add(Bidirectional(LSTM(100)))
model.add(Dropout(0.3))
model.add(Dense(1, activation = 'sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics =['accuracy'])
print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs = 10, batch_size=64)

Epoch 1/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 28s 32ms/step - accuracy: 0.8512 - loss: 0.3288 - val_accuracy: 0.8903 - val_loss: 0.2605
Epoch 2/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 24s 31ms/step - accuracy: 0.9032 - loss: 0.2325 - val_accuracy: 0.8929 - val_loss: 0.2561
Epoch 3/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 23s 31ms/step - accuracy: 0.9167 - loss: 0.2051 - val_accuracy: 0.8941 - val_loss: 0.2577
Epoch 4/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 23s 31ms/step - accuracy: 0.9239 - loss: 0.1864 - val_accuracy: 0.8943 - val_loss: 0.2614
Epoch 5/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 42s 33ms/step - accuracy: 0.9313 - loss: 0.1696 - val_accuracy: 0.8921 - val_loss: 0.2743
Epoch 6/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 24s 32ms/step - accuracy: 0.9381 - loss: 0.1544 - val_accuracy: 0.8938 - val_loss: 0.2919
Epoch 7/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 24s 32ms/step - accuracy: 0.9421 - loss: 0.1413 - val_accuracy: 0.8912 - val_loss: 0.3085
Epoch 8/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 24s 32ms/step - accuracy: 0.9471 - loss: 0.1302 - 

In [ ]:
y_pred = model.predict(X_test)

738/738 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step


In [ ]:
y_pred = np.where(y_pred >= 0.5, 1, 0)

In [ ]:
confusion_matrix(y_test, y_pred)

array([[10296,  1339],
       [ 1284, 10688]])

In [ ]:
accuracy_score(y_test, y_pred)

0.8888888888888888

In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.88      0.89     11635
           1       0.89      0.89      0.89     11972

    accuracy                           0.89     23607
   macro avg       0.89      0.89      0.89     23607
weighted avg       0.89      0.89      0.89     23607

